In [1]:
# 알아야 할 사실
# 영어 한글 1대1 글자 매핑이 안됨! 스타일만 옮길 수 있게끔

# GAN 추천 구조
# Style Encoder : 글리프 이미지 여러장 -> 스타일 벡터
# Generator : 한ㄱ르 글자 id, s -> 해당 한글 글리프 이미지
# Discriminator : 생성된 한글이 진짜 폰트 스타일 인지 판별

# 데이터 조건
# 입력 : 영어 글리프 이미지
# 정답 : 한글 글리프 이미지

In [2]:
import os
import random
import hashlib
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
import pandas as pd

In [3]:
# 폰트 경로 설정
FONT_DIR = Path(r"./filtered_fonts/google_valid_filtered") #폰트들이 들어있는 폴더 경로
OUT_DIR = Path(r"./gan_dataset")

IMG_SIZE = 128          # 한글 글리프 이미지 크기
GRID_CELL = 96          # 영어 그리드 셀 크기
GRID_COLS = 10          # 영어 그리드 열 개수
FONT_SIZE_GLYPH = 110   # 한글 글리프 렌더 폰트 크기
FONT_SIZE_GRID = 70     # 영어 그리드 렌더 폰트 크기
PADDING = 8

KOR_MIN_TEXT = (
    "가나더려모부쇼야져쵸켜튜프히"
)

KOR_CHARS = [c for c in KOR_MIN_TEXT] # 글자 단위

# 영어 입력(스타일 추출용) - 한 장 그리드로 만들기
ENG_CHARS = list("ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz")
ENG_CHARS += list(".,:;!?\"'()[]{}+-=*/@#%&0123456789")

# =========================
# 유틸
# =========================
SEED = 42
TRAIN_N = 66
VAL_N = 14
TEST_N = 14


def list_fonts(font_dir: Path):
    fonts = []
    for p in font_dir.iterdir():
        if p.suffix.lower() in [".ttf", ".otf", ".ttc", ".otc"]:
            fonts.append(p)
    fonts.sort()
    return fonts

def stable_id_from_path(p: Path) -> str:
    # 파일명만으로 충돌 가능성 있어서 해시로 안정 ID
    h = hashlib.sha1(str(p).encode("utf-8")).hexdigest()[:10]
    return f"{p.stem}_{h}"

def safe_load_font(font_path: Path, size: int):
    # PIL이 일부 폰트를 못 읽을 수 있어서 예외처리
    try:
        return ImageFont.truetype(str(font_path), size=size, layout_engine=ImageFont.Layout.BASIC)
    except Exception:
        # BASIC 옵션이 안 먹는 환경도 있어서 fallback
        return ImageFont.truetype(str(font_path), size=size)

def render_char_image(font_path: Path, char: str, out_path: Path, img_size=128, font_size=110):
    font = safe_load_font(font_path, font_size)
    img = Image.new("L", (img_size, img_size), color=255)  # 흰 배경
    draw = ImageDraw.Draw(img)

    # 텍스트 bbox로 중앙 정렬
    bbox = draw.textbbox((0, 0), char, font=font)
    w = bbox[2] - bbox[0]
    h = bbox[3] - bbox[1]
    x = (img_size - w) // 2 - bbox[0]
    y = (img_size - h) // 2 - bbox[1]

    draw.text((x, y), char, font=font, fill=0)  # 검정 글자
    out_path.parent.mkdir(parents=True, exist_ok=True)
    img.save(out_path)

def render_english_grid(font_path: Path, out_path: Path, chars, cell=96, cols=10, font_size=70, pad=8):
    font = safe_load_font(font_path, font_size)
    rows = (len(chars) + cols - 1) // cols
    W, H = cols * cell, rows * cell

    img = Image.new("L", (W, H), color=255)
    draw = ImageDraw.Draw(img)

    for i, ch in enumerate(chars):
        r = i // cols
        c = i % cols
        x0, y0 = c * cell, r * cell

        # cell 내부 중앙 정렬
        bbox = draw.textbbox((0, 0), ch, font=font)
        w = bbox[2] - bbox[0]
        h = bbox[3] - bbox[1]
        x = x0 + (cell - w) // 2 - bbox[0]
        y = y0 + (cell - h) // 2 - bbox[1]

        # 너무 타이트하면 pad 적용
        x = max(x0 + pad, min(x, x0 + cell - pad - w))
        y = max(y0 + pad, min(y, y0 + cell - pad - h))

        draw.text((x, y), ch, font=font, fill=0)

    out_path.parent.mkdir(parents=True, exist_ok=True)
    img.save(out_path)

def split_fonts(font_paths, seed=42):
    random.seed(seed)
    paths = font_paths[:]
    random.shuffle(paths)
    assert len(paths) == TRAIN_N + VAL_N + TEST_N, f"폰트 개수({len(paths)})가 94가 아님. 분할 수를 조정해야 함."
    train = paths[:TRAIN_N]
    val = paths[TRAIN_N:TRAIN_N + VAL_N]
    test = paths[TRAIN_N + VAL_N:]
    return train, val, test

# =========================
# 실행
# =========================
fonts = list_fonts(FONT_DIR)
print("폰트 개수:", len(fonts))

train_fonts, val_fonts, test_fonts = split_fonts(fonts, seed=SEED)

splits = [("train", train_fonts), ("val", val_fonts), ("test", test_fonts)]

records = []

for split_name, split_fonts_list in splits:
    for fp in split_fonts_list:
        font_id = stable_id_from_path(fp)

        # 저장 구조:
        # gan_dataset/{split}/{font_id}/input/english_grid.png
        # gan_dataset/{split}/{font_id}/target/{U+AC00}.png (글자별)
        base = OUT_DIR / split_name / font_id
        input_dir = base / "input"
        target_dir = base / "target"

        # 1) 영어 그리드 1장
        grid_path = input_dir / "english_grid.png"
        try:
            render_english_grid(fp, grid_path, ENG_CHARS, cell=GRID_CELL, cols=GRID_COLS, font_size=FONT_SIZE_GRID, pad=PADDING)
        except Exception as e:
            print(f"[영어그리드 실패] {fp.name}: {e}")
            continue

        # 2) 한글 최소 글리프들
        ok = 0
        for ch in KOR_CHARS:
            code = f"U+{ord(ch):04X}"
            out_path = target_dir / f"{code}.png"
            try:
                render_char_image(fp, ch, out_path, img_size=IMG_SIZE, font_size=FONT_SIZE_GLYPH)
                ok += 1
            except Exception:
                # 어떤 폰트는 특정 글리프가 없을 수 있으니 스킵
                pass

        records.append({
            "split": split_name,
            "font_id": font_id,
            "font_file": str(fp),
            "english_grid": str(grid_path),
            "korean_targets_rendered": ok,
            "korean_targets_total": len(KOR_CHARS),
        })

df = pd.DataFrame(records)
manifest_path = OUT_DIR / "manifest.csv"
OUT_DIR.mkdir(parents=True, exist_ok=True)
df.to_csv(manifest_path, index=False, encoding="utf-8-sig")

print("완료!")
print("manifest:", manifest_path)
print(df.groupby("split")[["korean_targets_rendered"]].mean())

폰트 개수: 94
완료!
manifest: gan_dataset\manifest.csv
       korean_targets_rendered
split                         
test                      14.0
train                     14.0
val                       14.0


In [4]:
import os
import random
import hashlib
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
import pandas as pd
import numpy as np

In [5]:
# =========================
# 폰트 경로 설정
# =========================
# 두 폴더를 리스트로 관리 → 자동으로 합산
FONT_DIRS = [
    Path(r"./filtered_fonts/google_valid_filtered"),    # 기존 94개
    Path(r"./filtered_fonts/new_opensource_filtered"),  # 추가 314개
]
OUT_DIR = Path(r"./gan_dataset")

IMG_SIZE       = 128   # 한글 글리프 이미지 크기
ENG_IMG_SIZE   = 128   # 영어 개별 글리프 이미지 크기
FONT_SIZE_KOR  = 110   # 한글 글리프 렌더 폰트 크기
FONT_SIZE_ENG  = 100   # 영어 글리프 렌더 폰트 크기

# MX-Font 입력용 한글 대표 14자
KOR_CHARS = list("가나더려모부쇼야져쵸켜튜프히")

# 영어 입력 후보 (특수문자, 숫자 제외 → A-Z, a-z)
ENG_POOL = list("ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz")

# 학습 시 DataLoader에서 랜덤 샘플링할 수 (참고용 상수)
ENG_SAMPLE_N = 16

# 분할 비율 (폰트 개수 상관없이 자동 계산 → assert 오류 없음)
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
# TEST = 나머지 0.15

# 빈 이미지 판단 기준 (픽셀 평균이 이 값 이상이면 글자 없음으로 판단)
BLANK_THRESHOLD = 250

SEED = 42

In [6]:
# =========================
# 유틸 함수
# =========================

def list_fonts(font_dirs):
    """여러 폴더에서 폰트 수집, MD5 해시로 완전 중복 제거"""
    fonts = []
    seen  = set()
    for d in font_dirs:
        if not d.exists():
            print(f"[경고] 폴더 없음: {d}")
            continue
        for p in sorted(d.iterdir()):
            if p.suffix.lower() in [".ttf", ".otf", ".ttc", ".otc"]:
                h = hashlib.md5(p.read_bytes()).hexdigest()
                if h not in seen:
                    seen.add(h)
                    fonts.append(p)
    fonts.sort(key=lambda p: p.name)
    return fonts


def stable_id_from_path(p: Path) -> str:
    h = hashlib.sha1(str(p).encode("utf-8")).hexdigest()[:10]
    return f"{p.stem}_{h}"


def safe_load_font(font_path: Path, size: int):
    try:
        return ImageFont.truetype(str(font_path), size=size,
                                  layout_engine=ImageFont.Layout.BASIC)
    except Exception:
        return ImageFont.truetype(str(font_path), size=size)


def render_char_centered(font, char: str, img_size: int) -> Image.Image:
    """단일 글자를 흰 배경 중앙에 렌더링 (L 모드)"""
    img  = Image.new("L", (img_size, img_size), color=255)
    draw = ImageDraw.Draw(img)
    bbox = draw.textbbox((0, 0), char, font=font)
    w = bbox[2] - bbox[0]
    h = bbox[3] - bbox[1]
    x = (img_size - w) // 2 - bbox[0]
    y = (img_size - h) // 2 - bbox[1]
    draw.text((x, y), char, font=font, fill=0)
    return img


def is_blank(img: Image.Image, threshold=250) -> bool:
    """픽셀 평균 >= threshold 이면 빈 이미지 (글자 없음)"""
    return np.array(img).mean() >= threshold


def split_fonts(font_paths, seed=42):
    """
    비율 기반 분할.
    기존 assert(고정 숫자) 제거 → 폰트 개수가 몇 개든 오류 없이 동작
    """
    rng   = random.Random(seed)
    paths = font_paths[:]
    rng.shuffle(paths)

    n       = len(paths)
    n_train = int(n * TRAIN_RATIO)       # 70%
    n_val   = int(n * VAL_RATIO)         # 15%
    # 나머지 전부 test → 반올림 오차를 test 쪽이 흡수

    train = paths[:n_train]
    val   = paths[n_train : n_train + n_val]
    test  = paths[n_train + n_val :]

    print(f"전체 폰트: {n}개  →  train: {len(train)}, val: {len(val)}, test: {len(test)}")
    return train, val, test

In [7]:
# =========================
# 실행
# =========================

# 저장 구조:
# gan_dataset/{split}/{font_id}/
#     input/   A.png, B.png, lower_a.png, ...  (영어 62자 개별 이미지 전체 저장)
#     target/  U+AC00.png, U+B098.png, ...     (한글 14자)
#
# ★ 학습 DataLoader에서 input/ 중 ENG_SAMPLE_N장을 랜덤 샘플링해서 Style Encoder에 입력

fonts = list_fonts(FONT_DIRS)
print(f"중복 제거 후 폰트 총합: {len(fonts)}개")

train_fonts, val_fonts, test_fonts = split_fonts(fonts, seed=SEED)
splits = [("train", train_fonts), ("val", val_fonts), ("test", test_fonts)]

records = []
skipped = []

for split_name, split_font_list in splits:
    for fp in split_font_list:
        font_id    = stable_id_from_path(fp)
        base       = OUT_DIR / split_name / font_id
        input_dir  = base / "input"
        target_dir = base / "target"

        # ── 1) 한글 폰트 유효성 먼저 체크 ────────────────────────
        try:
            kor_font = safe_load_font(fp, FONT_SIZE_KOR)
        except Exception as e:
            skipped.append({"font": fp.name, "reason": f"폰트 로드 실패: {e}"})
            continue

        probe = render_char_centered(kor_font, KOR_CHARS[0], IMG_SIZE)
        if is_blank(probe, BLANK_THRESHOLD):
            skipped.append({"font": fp.name, "reason": "한글 글리프 없음 (빈 이미지)"})
            continue

        # ── 2) 영어 개별 글리프 저장 (62자 전체) ─────────────────
        try:
            eng_font = safe_load_font(fp, FONT_SIZE_ENG)
        except Exception as e:
            skipped.append({"font": fp.name, "reason": f"영어 폰트 로드 실패: {e}"})
            continue

        input_dir.mkdir(parents=True, exist_ok=True)
        eng_saved = []
        for ch in ENG_POOL:
            img = render_char_centered(eng_font, ch, ENG_IMG_SIZE)
            if is_blank(img, BLANK_THRESHOLD):
                continue
            # 대소문자 파일명 충돌 방지: 소문자는 lower_ 접두어
            fname    = f"lower_{ch}.png" if ch.islower() else f"{ch}.png"
            img.save(input_dir / fname)
            eng_saved.append(fname)

        if len(eng_saved) < ENG_SAMPLE_N:
            skipped.append({"font": fp.name,
                            "reason": f"영어 글리프 부족 ({len(eng_saved)}장 < {ENG_SAMPLE_N})"})
            continue

        # ── 3) 한글 대표 14자 저장 ───────────────────────────────
        target_dir.mkdir(parents=True, exist_ok=True)
        kor_ok = 0
        for ch in KOR_CHARS:
            code     = f"U+{ord(ch):04X}"
            out_path = target_dir / f"{code}.png"
            try:
                img = render_char_centered(kor_font, ch, IMG_SIZE)
                if is_blank(img, BLANK_THRESHOLD):
                    continue
                img.save(out_path)
                kor_ok += 1
            except Exception:
                pass

        records.append({
            "split"              : split_name,
            "font_id"            : font_id,
            "font_file"          : str(fp),
            "eng_glyphs_saved"   : len(eng_saved),
            "kor_glyphs_rendered": kor_ok,
            "kor_glyphs_total"   : len(KOR_CHARS),
            "input_dir"          : str(input_dir),
            "target_dir"         : str(target_dir),
        })

# ── 저장 ──────────────────────────────────────────────────────────
OUT_DIR.mkdir(parents=True, exist_ok=True)
df = pd.DataFrame(records)
df.to_csv(OUT_DIR / "manifest.csv", index=False, encoding="utf-8-sig")

if skipped:
    pd.DataFrame(skipped).to_csv(OUT_DIR / "skipped.csv", index=False, encoding="utf-8-sig")
    print(f"제외 폰트: {len(skipped)}개  → skipped.csv 확인")

print(f"\n=== 완료 ===")
print(f"유효 폰트: {len(records)}개 / 전체: {len(fonts)}개")
print("\n분할별 평균 렌더 수:")
print(df.groupby("split")[["kor_glyphs_rendered", "eng_glyphs_saved"]].mean())

중복 제거 후 폰트 총합: 403개
전체 폰트: 403개  →  train: 282, val: 60, test: 61
제외 폰트: 10개  → skipped.csv 확인

=== 완료 ===
유효 폰트: 393개 / 전체: 403개

분할별 평균 렌더 수:
       kor_glyphs_rendered  eng_glyphs_saved
split                                       
test             14.000000         49.964912
train            14.000000         50.292419
val              13.949153         49.491525
